# Wakacje.pl availability checker

Check whether a specific wakacje.pl offer variant is bookable using their public APIs.

**Flow**

1. Parse the offer URL (`od-YYYY-MM-DD,N-dni,BOARD,z-departure,Ndorosle` selector).
2. Load static context from the page `__NEXT_DATA__` (hotel, tour operator, geo IDs).
3. Resolve the concrete room variant via `POST /v2/api/getCalculatorOfferVariants/{offerId}`.
4. Confirm live availability via `GET /v2/api/checkOfferAvailability`.

**Unavailable configurations**

When step 3 returns `"offers": []`, the exact URL configuration is not sold (same as the on-site
“Oferta w tej konfiguracji jest niedostępna” message). The checker reports `available=False`
without calling step 4.

In [39]:
from __future__ import annotations

import json
import re
import unicodedata
from dataclasses import dataclass
from typing import Any
from urllib.parse import urlparse

import httpx
from bs4 import BeautifulSoup

BASE_URL = "https://www.wakacje.pl"
DEFAULT_ADULT_BIRTH_DATE = "1988-01-01"
USER_AGENT = (
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
    "AppleWebKit/605.1.15 (KHTML, like Gecko) Version/26.5 Safari/605.1.15"
)

SERVICE_MAP = {
    "AI": 1,
    "ALL-INCLUSIVE": 1,
    "ALLINCLUSIVE": 1,
    "HB": 2,
    "BB": 3,
    "WL": 4,
    "FB": 6,
}

# Airport-specific slugs need exact wakacje city IDs (generic "Warszawa" != Chopin/Modlin).
DEPARTURE_SLUG_TO_ID = {
    "warszawy-chopin": 10119,
    "warszawy-modlin": 9758,
    "wroclawia": 256,
    "krakowa": 2696,
    "gdanska": 2880,
    "katowic": 2622,
    "poznania": 2632,
    "rzeszowa": 1909,
    "bydgoszczy": 269,
    "lodzi": 2654,
    "szczecina": 2904,
    "berlina": 3301,
}

# Fallback prefix matching for slugs not in DEPARTURE_SLUG_TO_ID.
DEPARTURE_SLUG_PREFIXES = {
    "warszawy": "warszawa",
    "warszawa": "warszawa",
    "wroclaw": "wroclaw",
    "krakow": "krakow",
    "gdansk": "gdansk",
    "katowice": "katowice",
    "poznan": "poznan",
    "rzeszow": "rzeszow",
    "berlin": "berlin",
}


def _strip_accents(value: str) -> str:
    return "".join(
        char
        for char in unicodedata.normalize("NFD", value)
        if unicodedata.category(char) != "Mn"
    )


def _json_headers(referer: str) -> dict[str, str]:
    return {
        "accept": "application/json",
        "content-type": "application/json",
        "origin": BASE_URL,
        "referer": referer,
        "user-agent": USER_AGENT,
    }

In [40]:
@dataclass(frozen=True)
class OfferSelector:
    offer_id: int
    departure_date: str
    duration_nights: int
    board_code: str
    departure_slug: str
    adults: int


@dataclass(frozen=True)
class OfferContext:
    offer_id: int
    hotel_id: int
    tour_operator_id: int
    city_id: int
    country_id: int
    region_id: int
    name: str
    tour_op_code: str | None = None


@dataclass(frozen=True)
class OfferVariant:
    offer_hash: str
    provider_code: str
    room_desc: str
    total_price: int
    currency: str
    departure_place: str


@dataclass(frozen=True)
class AvailabilityResult:
    url: str
    selector: OfferSelector
    context: OfferContext
    variant: OfferVariant | None
    available: bool
    status: str
    price: int | None
    currency: str | None
    departure_place: str | None
    reason: str | None
    raw: dict[str, Any] | None


def _normalize_board_code(raw_board: str) -> str:
    normalized = raw_board.upper().replace("-", "")
    aliases = {
        "ALLINCLUSIVE": "ALL-INCLUSIVE",
        "AI": "AI",
        "HB": "HB",
        "BB": "BB",
        "WL": "WL",
        "FB": "FB",
    }
    if normalized in aliases:
        return aliases[normalized]
    if raw_board.upper() in SERVICE_MAP:
        return raw_board.upper()
    raise ValueError(f"Unsupported board code: {raw_board!r}")


def parse_offer_url(url: str) -> OfferSelector:
    """Parse wakacje offer URL path + `od-...` selector."""
    parsed = urlparse(url)
    offer_id_match = re.search(r"-(\d+)\.html", parsed.path)
    if not offer_id_match:
        raise ValueError(f"Could not parse offer id from URL path: {parsed.path}")

    selector_match = re.search(r"od-([^?&]+)", parsed.query)
    if not selector_match:
        raise ValueError("URL is missing the `od-...` offer selector query part")

    selector = re.split(r"utm_", selector_match.group(1))[0].rstrip(",")
    parts = selector.split(",")
    if len(parts) < 5:
        raise ValueError(f"Unexpected selector format: {selector!r}")

    duration_match = re.search(r"(\d+)", parts[1])
    adults_match = re.search(r"(\d+)", parts[4])
    if not duration_match or not adults_match:
        raise ValueError(f"Could not parse duration/adults from selector: {selector!r}")

    board_code = _normalize_board_code(parts[2])

    return OfferSelector(
        offer_id=int(offer_id_match.group(1)),
        departure_date=parts[0],
        duration_nights=int(duration_match.group(1)),
        board_code=board_code,
        departure_slug=parts[3],
        adults=int(adults_match.group(1)),
    )

In [41]:
def _page_headers() -> dict[str, str]:
    return {
        "user-agent": USER_AGENT,
        "accept-language": "pl-PL,pl;q=0.9",
        "accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    }


def _extract_offer_details(page_json: dict[str, Any]) -> dict[str, Any]:
    store = page_json["props"]["stores"]["storeOfferDetails"]
    offer_details = dict(store.get("offerDetails") or {})

    if offer_details.get("tourOpCode"):
        return offer_details

    for entry in store.get("offerDetailsList") or []:
        if not isinstance(entry, list) or len(entry) < 2:
            continue
        cached = entry[1]
        if not isinstance(cached, dict):
            continue
        cached_data = (
            cached.get("data") if isinstance(cached.get("data"), dict) else cached
        )
        if cached_data.get("tourOpCode"):
            return {**offer_details, **cached_data}

    return offer_details


def _require_field(offer_details: dict[str, Any], field: str) -> Any:
    if field in offer_details and offer_details[field] is not None:
        return offer_details[field]
    available = ", ".join(sorted(offer_details))
    raise ValueError(
        f"Offer page JSON is missing {field!r}; available fields: {available}"
    )


def fetch_offer_context(client: httpx.Client, url: str) -> OfferContext:
    response = client.get(url, headers=_page_headers())
    response.raise_for_status()

    next_data = BeautifulSoup(response.text, "html.parser").find(
        "script", id="__NEXT_DATA__"
    )
    if not next_data or not next_data.string:
        raise ValueError("Offer page did not contain __NEXT_DATA__")

    offer_details = _extract_offer_details(json.loads(next_data.string))
    return OfferContext(
        offer_id=_require_field(offer_details, "offerId"),
        hotel_id=_require_field(offer_details, "hotelId"),
        tour_op_code=offer_details.get("tourOpCode"),
        tour_operator_id=_require_field(offer_details, "tourOperatorId"),
        city_id=_require_field(offer_details, "cityId"),
        country_id=_require_field(offer_details, "countryId"),
        region_id=_require_field(offer_details, "regionId"),
        name=_require_field(offer_details, "name"),
    )


def resolve_departure_city_id(
    client: httpx.Client,
    offer_id: int,
    departure_slug: str,
) -> tuple[int, str]:
    raw_slug = _strip_accents(departure_slug.removeprefix("z-")).lower()
    if raw_slug in DEPARTURE_SLUG_TO_ID:
        city_id = DEPARTURE_SLUG_TO_ID[raw_slug]
        label = raw_slug.replace("-", " ").title()
        return city_id, label

    response = client.post(
        f"{BASE_URL}/v2/api/offerConfiguratorV2/filters",
        json={
            "offerId": offer_id,
            "departureDate": [],
            "duration": [],
            "adults": 2,
            "kidsAges": [],
            "service": [],
            "transportType": [],
            "departurePlace": [],
            "providerIds": [],
            "isConfiguratorToFilterEnabled": True,
        },
        headers=_json_headers(f"{BASE_URL}/"),
    )
    response.raise_for_status()
    places = response.json()["data"]["departurePlaces"]

    city_key = DEPARTURE_SLUG_PREFIXES.get(
        raw_slug,
        DEPARTURE_SLUG_PREFIXES.get(raw_slug.split("-")[0], raw_slug),
    )

    for place in places:
        label = _strip_accents(place["label"]).lower()
        if city_key in label or label.startswith(city_key[:4]):
            return place["value"], place["label"]

    known = ", ".join(place["label"] for place in places)
    raise ValueError(
        f"Unknown departure slug {departure_slug!r}; known airports: {known}"
    )

In [42]:
def fetch_offer_variants(
    client: httpx.Client,
    url: str,
    selector: OfferSelector,
    context: OfferContext,
) -> tuple[list[OfferVariant], str, dict[str, Any]]:
    departure_city_id, departure_label = resolve_departure_city_id(
        client,
        selector.offer_id,
        selector.departure_slug,
    )

    payload: dict[str, Any] = {
        "adults": selector.adults,
        "kids": 0,
        "infants": 0,
        "kidsAges": [],
        "serviceId": SERVICE_MAP[selector.board_code],
        "duration": selector.duration_nights,
        "departureDate": selector.departure_date,
        "transportId": 1,
        "departureCityId": departure_city_id,
        "departureCityCode": "WMI",
        "hotelId": context.hotel_id,
        "tourId": context.tour_operator_id,
        "cruiseId": 0,
        "roundTripId": 0,
        "isAlternativeRoom": False,
        "isOffer77": False,
    }
    if context.tour_op_code:
        payload["tourOp"] = context.tour_op_code

    response = client.post(
        f"{BASE_URL}/v2/api/getCalculatorOfferVariants/{selector.offer_id}",
        json=payload,
        headers=_json_headers(url),
    )
    response.raise_for_status()
    body = response.json()
    offers = body["data"]["offers"]

    variants = [
        OfferVariant(
            offer_hash=offer["id"],
            provider_code=offer.get("providerCode")
            or offer.get("tourOp")
            or context.tour_op_code
            or "",
            room_desc=offer["roomDesc"],
            total_price=offer["totalPrice"],
            currency=offer["priceCurrency"],
            departure_place=offer["departStart"]["name"],
        )
        for offer in offers
    ]
    return variants, departure_label, body


def check_variant_availability(
    client: httpx.Client,
    url: str,
    selector: OfferSelector,
    context: OfferContext,
    variant: OfferVariant,
) -> dict[str, Any]:
    provider_code = variant.provider_code or context.tour_op_code
    if not provider_code:
        raise ValueError("Could not resolve provider code for availability check")

    params: dict[str, str | int | bool] = {
        "providerCode": provider_code,
        "offerHash": variant.offer_hash,
        "offerType": "tour",
        "includeTfgService": "true",
        "isAlternativeRoom": "false",
        "cityId": context.city_id,
        "countryId": context.country_id,
        "regionId": context.region_id,
    }

    for index in range(selector.adults):
        params[f"participantsObject[participants][{index}][userAllocateId]"] = index + 1
        params[f"participantsObject[participants][{index}][birthDate]"] = (
            DEFAULT_ADULT_BIRTH_DATE
        )
        params[f"participantsObject[participants][{index}][type]"] = "adult"

    custom_headers = json.dumps(
        {
            "Page-Source": "PO",
            "Tour-Operator-Code": provider_code,
            "Tour-Operator-Id": str(context.tour_operator_id),
            "Object-Id": str(context.hotel_id),
        }
    )

    response = client.get(
        f"{BASE_URL}/v2/api/checkOfferAvailability",
        params=params,
        headers={
            "accept": "application/json",
            "referer": url,
            "customHeaders": custom_headers,
            "user-agent": USER_AGENT,
        },
    )
    response.raise_for_status()
    return response.json()

In [43]:
def check_offer_availability(url: str, *, room_index: int = 0) -> AvailabilityResult:
    """Return live availability for the offer variant encoded in the URL."""
    selector = parse_offer_url(url)

    with httpx.Client(
        timeout=30.0,
        follow_redirects=True,
        headers=_page_headers(),
    ) as client:
        context = fetch_offer_context(client, url)
        variants, departure_label, variants_payload = fetch_offer_variants(
            client,
            url,
            selector,
            context,
        )

        if not variants:
            return AvailabilityResult(
                url=url,
                selector=selector,
                context=context,
                variant=None,
                available=False,
                status="UNAVAILABLE",
                price=None,
                currency=None,
                departure_place=departure_label,
                reason=(
                    "No room variants for this exact configuration "
                    f"({selector.departure_date}, {selector.duration_nights} nights, "
                    f"{selector.board_code}, {selector.departure_slug})"
                ),
                raw=variants_payload,
            )

        if room_index < 0 or room_index >= len(variants):
            raise IndexError(
                f"room_index {room_index} out of range (found {len(variants)} variants)"
            )

        variant = variants[room_index]
        payload = check_variant_availability(client, url, selector, context, variant)

    data = payload.get("data") or {}
    return AvailabilityResult(
        url=url,
        selector=selector,
        context=context,
        variant=variant,
        available=bool(data.get("availability")),
        status=str(data.get("status", "UNKNOWN")),
        price=data.get("price"),
        currency=data.get("currency"),
        departure_place=data.get("departureStringValue"),
        reason=None,
        raw=payload,
    )


def print_availability(result: AvailabilityResult) -> None:
    print(f"{result.context.name} ({result.context.offer_id})")
    print(
        f"  selector: {result.selector.departure_date}, "
        f"{result.selector.duration_nights} nights, "
        f"{result.selector.board_code}, "
        f"{result.selector.departure_slug}, "
        f"{result.selector.adults} adults"
    )
    if result.variant is None:
        print(f"  available: {result.available} | status: {result.status}")
        print(f"  reason: {result.reason}")
        return

    print(
        f"  room: {result.variant.room_desc} ({result.variant.total_price} {result.variant.currency})"
    )
    print(
        f"  available: {result.available} | status: {result.status} | "
        f"live price: {result.price} {result.currency} | departure: {result.departure_place}"
    )

In [45]:
TEST_URLS = [
    "https://www.wakacje.pl/oferty/hiszpania/fuerteventura/morro-jable/ifa-altamarena-790321.html?od-2026-06-08,7-dni,all-inclusive,z-wroclawia,2dorosleutm_source=travellead&utm_medium=cps&utm_campaign=2933-t-HolidayPicker&a_cid=11111111&a_aid=2933",
    "https://www.wakacje.pl/oferty/chorwacja/istria/vrsar/maistra-select-funtana-vrsar-843652.html?od-2026-06-10,5-dni,all-inclusive,z-katowic,2dorosleutm_source=travellead&utm_medium=cps&utm_campaign=2933-t-HolidayPicker&a_cid=11111111&a_aid=2933",
    # Available configuration
    "https://www.wakacje.pl/oferty/malta/wyspa-malta/bugibba/cardor-holiday-complex-1058607.html?od-2026-09-06,5-dni,FB,z-wroclawia,2dorosle&utm_source=travellead",
    # Unavailable configuration (site shows "Oferta w tej konfiguracji jest niedostępna")
    "https://www.wakacje.pl/oferty/egipt/hurghada/hurghada/blend-club-aqua-resort-ex-golden-five-club-745287.html?od-2026-06-11,7-dni,all-inclusive,z-warszawy-chopin,2dorosle&utm_source=travellead",
]

for test_url in TEST_URLS:
    result = check_offer_availability(test_url)
    print_availability(result)
    print()

IFA Altamarena (790321)
  selector: 2026-06-08, 7 nights, ALL-INCLUSIVE, z-wroclawia, 2 adults
  available: False | status: UNAVAILABLE
  reason: No room variants for this exact configuration (2026-06-08, 7 nights, ALL-INCLUSIVE, z-wroclawia)

Maistra Select Funtana (Vrsar) (843652)
  selector: 2026-06-10, 5 nights, ALL-INCLUSIVE, z-katowic, 2 adults
  available: False | status: UNAVAILABLE
  reason: No room variants for this exact configuration (2026-06-10, 5 nights, ALL-INCLUSIVE, z-katowic)

Cardor Holiday Complex (1058607)
  selector: 2026-09-06, 5 nights, FB, z-wroclawia, 2 adults
  room: Ap_dts standard apartment (typeb) (3996 PLN)
  available: True | status: OK | live price: 3996 PLN | departure: Wrocław

Blend Club Aqua Resort (ex. Golden Five Club) (745287)
  selector: 2026-06-11, 7 nights, ALL-INCLUSIVE, z-warszawy-chopin, 2 adults
  available: False | status: UNAVAILABLE
  reason: No room variants for this exact configuration (2026-06-11, 7 nights, ALL-INCLUSIVE, z-warszawy-